# 面试问题：浏览器 Agent 怎样用 DOM 快照、稳定引用与后置条件避免点错页面？

**一句话回答。** 浏览器 Agent 不应直接根据旧 HTML 或模型生成的任意 selector 执行动作。每轮 observation 要生成带 page/session/version 的精简 DOM 或 accessibility snapshot，为可交互元素分配本轮稳定引用；action 必须绑定 snapshot version、元素语义、风险等级和预期后置条件，执行前重验当前页面，执行后用 URL、DOM/业务状态回读确认结果。

本 Notebook 以虚构页面和 Python 状态机演示 observe–ground–act–verify。它不启动浏览器、不访问网站，也不把小数据断言宣传成真实网页上的成功率或安全保证。

**资料入口。** [WebArena](https://arxiv.org/abs/2307.13854) 构建了可复现的真实网站环境并以功能正确性评测长程网页任务；[BrowserGym](https://arxiv.org/abs/2412.05467) 提供统一的网页 Agent 观察与动作研究环境。

In [ ]:
question = "浏览器 Agent DOM 快照与陈旧动作验证"  # 执行本行的状态、计算或校验逻辑。
assert "DOM" in question  # 执行本行的状态、计算或校验逻辑。
assert 12 // 4 == 3  # 执行本行的状态、计算或校验逻辑。
assert True  # 执行本行的状态、计算或校验逻辑。

## 1. Observation 是有版本的环境证据

完整 DOM 很长，包含脚本、样式和大量与任务无关的节点；只给截图又可能丢失可访问名称与控件状态。生产系统通常组合 URL、页面标题、可访问树、可见元素、表单值和截图，并为快照记录 session/page/version。模型看到的是某一时刻的证据，不是永远有效的页面事实。

In [ ]:
snapshot = {"session": "s-1", "page": "checkout", "version": 7, "url": "https://shop.example/checkout", "elements": [{"ref": "e1", "role": "button", "name": "提交订单", "enabled": True}, {"ref": "e2", "role": "link", "name": "返回购物车", "enabled": True}]}  # 执行本行的状态、计算或校验逻辑。
assert snapshot["version"] == 7  # 执行本行的状态、计算或校验逻辑。
assert len(snapshot["elements"]) == 2  # 执行本行的状态、计算或校验逻辑。
assert snapshot["elements"][0]["name"] == "提交订单"  # 执行本行的状态、计算或校验逻辑。

## 2. Grounding 先按语义过滤，再交给模型选择

元素引用应由运行时根据本轮 snapshot 生成，模型只从允许集合中选择。role、可访问名称、可见性、enabled 和唯一性比脆弱的 CSS 路径更适合做教学合同；如果同名按钮不唯一，系统应请求更多上下文或澄清，而不是默认点第一个。引用只在绑定的页面版本内有效。

In [ ]:
def ground(snapshot_value, role, name):  # 执行本行的状态、计算或校验逻辑。
    return [item for item in snapshot_value["elements"] if item["role"] == role and item["name"] == name and item["enabled"]]  # 执行本行的状态、计算或校验逻辑。
matches = ground(snapshot, "button", "提交订单")  # 执行本行的状态、计算或校验逻辑。
assert [item["ref"] for item in matches] == ["e1"]  # 执行本行的状态、计算或校验逻辑。
assert ground(snapshot, "button", "不存在") == []  # 执行本行的状态、计算或校验逻辑。
assert ground(snapshot, "link", "返回购物车")[0]["ref"] == "e2"  # 执行本行的状态、计算或校验逻辑。

## 3. Action 合同绑定快照、意图与后置条件

模型提出的 click/type/submit 只是候选动作。运行时要保存 session、snapshot version、element ref、动作类型、用户意图摘要、风险级别、幂等键和 expected postcondition。这样审批、执行与审计针对的是同一个精确动作；若其中任一字段变化，就不能沿用旧批准或旧引用。

In [ ]:
action = {"session": "s-1", "snapshot_version": 7, "ref": "e1", "type": "click", "intent": "提交当前购物车订单", "risk": "high", "idempotency_key": "submit-cart-42", "expected": {"page": "confirmation", "status": "created"}}  # 执行本行的状态、计算或校验逻辑。
assert action["snapshot_version"] == snapshot["version"]  # 执行本行的状态、计算或校验逻辑。
assert action["ref"] == matches[0]["ref"]  # 执行本行的状态、计算或校验逻辑。
assert action["risk"] == "high"  # 执行本行的状态、计算或校验逻辑。

## 4. 执行前必须重验页面版本和元素语义

网页可能因导航、弹窗、A/B 实验、异步刷新或其他操作者而变化。旧 ref 在新快照中可能指向另一个节点，即使坐标和 selector 仍可解析也不能说明语义一致。executor 应核对 session、page/version、目标 role/name、enabled、风险审批与当前业务状态；陈旧动作应返回 stale_observation 并触发重新观察。

In [ ]:
def executable(action_value, current, approved):  # 执行本行的状态、计算或校验逻辑。
    refs = {item["ref"] for item in current["elements"] if item["enabled"]}  # 执行本行的状态、计算或校验逻辑。
    return action_value["session"] == current["session"] and action_value["snapshot_version"] == current["version"] and action_value["ref"] in refs and (action_value["risk"] != "high" or approved)  # 执行本行的状态、计算或校验逻辑。
assert executable(action, snapshot, True)  # 执行本行的状态、计算或校验逻辑。
assert not executable(action, {**snapshot, "version": 8}, True)  # 执行本行的状态、计算或校验逻辑。
assert not executable(action, snapshot, False)  # 执行本行的状态、计算或校验逻辑。

## 5. 副作用执行需要幂等账本

浏览器 click 本身通常没有端到端 exactly-once 语义。网络超时后重新点击提交按钮可能创建重复订单，所以执行层要把幂等键传给可控后端，或在重试前查询业务状态。教学状态机将同一个 idempotency key 映射到同一确认结果，展示“页面重试”和“业务重复”必须分开处理。

In [ ]:
ledger = {}  # 执行本行的状态、计算或校验逻辑。
def submit(action_value, ledger_value):  # 执行本行的状态、计算或校验逻辑。
    if action_value["idempotency_key"] not in ledger_value: ledger_value[action_value["idempotency_key"]] = {"order_id": "o-42", "status": "created", "page": "confirmation", "version": 8}  # 执行本行的状态、计算或校验逻辑。
    return ledger_value[action_value["idempotency_key"]]  # 执行本行的状态、计算或校验逻辑。
result = submit(action, ledger)  # 执行本行的状态、计算或校验逻辑。
assert result["order_id"] == "o-42"  # 执行本行的状态、计算或校验逻辑。
assert submit(action, ledger) is result  # 执行本行的状态、计算或校验逻辑。
assert len(ledger) == 1  # 执行本行的状态、计算或校验逻辑。

## 6. 工具返回成功后仍要验证权威后置条件

浏览器驱动返回 click 成功，只说明事件已发送；页面可能校验失败、跳到登录页或创建了错误对象。postcondition verifier 应从新页面或业务 API 回读，检查目标 page/status、订单主体、数量与非目标状态未变。模型生成的自然语言“已下单”不能代替这个确定性 oracle。

In [ ]:
def verify_postcondition(action_value, observed):  # 执行本行的状态、计算或校验逻辑。
    return all(observed.get(key) == value for key, value in action_value["expected"].items()) and observed.get("order_id") is not None  # 执行本行的状态、计算或校验逻辑。
assert verify_postcondition(action, result)  # 执行本行的状态、计算或校验逻辑。
assert not verify_postcondition(action, {**result, "status": "failed"})  # 执行本行的状态、计算或校验逻辑。
assert not verify_postcondition(action, {"page": "confirmation", "status": "created"})  # 执行本行的状态、计算或校验逻辑。

## 7. 页面差分降低上下文，但不能丢失安全状态

长任务可只发送相邻快照的新增、删除和属性变化，减少 token；但当前 URL、对话框、表单值、风险控件和 action 目标必须保留。差分要绑定 base version，缺失任一中间版本就重新抓完整快照。否则 Agent 可能在不可见弹窗或已变更表单上继续旧计划。

In [ ]:
next_snapshot = {"session": "s-1", "page": "confirmation", "version": 8, "url": "https://shop.example/confirmation", "elements": [{"ref": "e9", "role": "heading", "name": "订单已创建", "enabled": True}]}  # 执行本行的状态、计算或校验逻辑。
diff = {"base": 7, "target": 8, "page_changed": snapshot["page"] != next_snapshot["page"], "added_refs": {"e9"}, "removed_refs": {"e1", "e2"}}  # 执行本行的状态、计算或校验逻辑。
assert diff["base"] + 1 == diff["target"]  # 执行本行的状态、计算或校验逻辑。
assert diff["page_changed"]  # 执行本行的状态、计算或校验逻辑。
assert "e1" in diff["removed_refs"]  # 执行本行的状态、计算或校验逻辑。

## 8. 评测要看功能状态、安全路径与恢复能力

网页 Agent 的指标至少拆为 grounding accuracy、action validity、functional success、stale-action block、unsafe attempt、步骤/成本和恢复率。每个任务需固定初始网站快照与账号权限，用执行式 grader 检查最终状态；只按最终文本关键词打分会把“声称完成”误判为真正完成。

In [ ]:
metrics = {"grounded": 3, "valid_actions": 2, "functional_success": 1, "stale_blocked": 1, "unsafe_executed": 0}  # 执行本行的状态、计算或校验逻辑。
assert metrics["grounded"] >= metrics["valid_actions"]  # 执行本行的状态、计算或校验逻辑。
assert metrics["stale_blocked"] == 1  # 执行本行的状态、计算或校验逻辑。
assert metrics["unsafe_executed"] == 0  # 执行本行的状态、计算或校验逻辑。

## 面试总结

回答浏览器 Agent 时可按 observe、ground、authorize、act、verify、recover 六步展开。最关键的边界是：DOM/截图是带版本的观察，元素引用不是永久地址；执行成功不是业务成功；高风险动作必须绑定精确参数和当前页面；最终结果应由可复现环境状态验证。真实部署还需浏览器沙箱、秘密隔离、下载/上传策略、反钓鱼、人工审批和跨站权限控制。